# प्रॉम्प्ट अभियांत्रणाची ओळख
प्रॉम्प्ट अभियांत्रण म्हणजे नैसर्गिक भाषा प्रक्रिया कार्यांसाठी प्रॉम्प्ट तयार करणे आणि त्यांचे अनुकूलन करणे. यात योग्य प्रॉम्प्ट निवडणे, त्यांचे पॅरामिटर समायोजित करणे, आणि त्यांच्या कामगिरीचे मूल्यांकन करणे यांचा समावेश होतो. प्रॉम्प्ट अभियांत्रण हे NLP मॉडेल्समध्ये उच्च अचूकता आणि कार्यक्षमता साध्य करण्यासाठी अत्यंत महत्वाचे आहे. या विभागात, आपण OpenAI मॉडेल्सचा वापर करून प्रॉम्प्ट अभियांत्रणाच्या मूलभूत गोष्टींचा अभ्यास करू.  


### व्यायाम 1: टोकनायझेशन
OpenAI कडून ओपन-सोर्स फास्ट टोकनायझर tiktoken वापरून टोकनायझेशन एक्सप्लोर करा
अधिक उदाहरणांसाठी [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) पहा.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### अभ्यास 2: मायक्रोसॉफ्ट फाउंड्री मॉडेल्स की सेटअपची पडताळणी करा

> **टीप:** GitHub Models जुलै 2026 च्या शेवटी बंद होत आहे. या अभ्यासात त्याऐवजी [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) वापरले आहे, जे त्याच विनामूल्य प्रयत्नासाठी मॉडेल कॅटलॉग आणि Azure AI Inference SDK अनुभव प्रदान करते.

खालील कोड चालवा जेणेकरून तुमचा Microsoft Foundry Models एंडपॉइंट योग्यरित्या सेटअप झाला आहे की नाही हे तपासता येईल. तो कोड एक सोपा बेसिक प्रॉम्प्ट वापरतो आणि पूर्णतेची पडताळणी करतो. इनपुट `oh say can you see` या ओळीचे पूर्ण होणे `by the dawn's early light..` या सारखे अपेक्षित आहे.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### व्यायाम ३: बनावट
जेव्हा आपण LLM ला अशा विषयाबद्दल प्रॉम्प्टसाठी पूर्णता परत करण्यास सांगता जे अस्तित्वात नसेल, किंवा अशा विषयांबद्दल जे त्याला माहित नसतील कारण ते त्याच्या आधीच्या प्रशिक्षण डेटासेटबाहेर आहेत (अधिक अलीकडील), तेव्हा काय घडते ते अन्वेषण करा. वेगळा प्रॉम्प्ट वापरल्यास किंवा वेगळा मॉडेल वापरल्यास प्रतिसाद कसा बदलतो ते पहा.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### व्यायाम ४: सूचना आधारित 
प्राथमिक सामग्री सेट करण्यासाठी "text" व्हेरिएबल वापरा 
आणि त्या प्राथमिक सामग्रीसंबंधित निर्देश देण्यासाठी "prompt" व्हेरिएबल वापरा.

येथे आम्ही मॉडेलला दुसऱ्या वर्गाच्या विद्यार्थ्याला मजकूराचा सारांश देण्यास सांगतो


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### व्यायाम 5: जटिल प्रॉम्प्ट  
अशी विनंती करा ज्यात सिस्टम, वापरकर्ता आणि सहाय्यक संदेश असतील  
सिस्टम सहाय्यक संदर्भ सेट करतो  
वापरकर्ता आणि सहाय्यक संदेश बहु-चाली संभाषण संदर्भ प्रदान करतात  

लक्षात घ्या की सहाय्यक व्यक्तिमत्व सिस्टम संदर्भात "विक्षिप्त" म्हणून सेट केले आहे.  
वेगळे व्यक्तिमत्व संदर्भ वापरून पाहा. किंवा संदेशांच्या इनपुट/आउटपुट मालिकेत बदल करून पाहा  


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### व्यायाम: तुमची अंतर्ज्ञान अन्वेषण करा
वरील उदाहरणे तुम्हाला काही नमुने देतात ज्यांचा वापर तुम्ही नवीन प्रॉम्प्ट तयार करण्यासाठी करू शकता (सोपे, क्लिष्ट, सूचनात्मक इत्यादी) - काही इतर कल्पना जसे की उदाहरणे, संकेत आणि बरेच काही यांचा अभ्यास करण्यासाठी इतर व्यायाम तयार करण्याचा प्रयत्न करा.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
